# 重要名詞定義

這邊不懂的話, 下面都不用看了

- main Agent == primary Agent
- main Agent 指的就是, 每次與 AI 展開溝通的對話窗口
- `frontmatter` 指的是, "---" 包起來的 key-value pair 區塊. 這部分包含:
  - name        (REAUIRED)
  - description (REAUIRED)
  - tools       (OPTIONAL)
  - color       (OPTIONAL)
  - model       (OPTIONAL)
  - .....       (OPTIONAL)
- `Instruction` 指的是 frontmatter 以外的部分, 這裡頭的情境為 Agent(sub-agent) 的 **System Prompt** - `具備引導作用但是不具備強制力, 一切的決定都是 LLM 頭腦風暴後決定`

# AI Agent

- 所有 `Agentic System` 關注的焦點都圍繞在: `Context` / `Model` / `Prompt`
- IndyDevDan Code Repository - https://github.com/disler/claude-code-hooks-mastery


✅ [My Claude Code Sub Agents BUILD THEMSELVES](https://www.youtube.com/watch?v=7B2HJr0Y68g)

**影片重點**

作者嘗試分享, 如何搭建 Agents 團隊, 讓 main Agent 委派任務給 agents 進行(任務專業化分工), 能有效減少 main Agent 上下文污染 && 節省 token.
良好的 *Chained Sub Agent Orchestration* 應該要像是這樣子:

```
                     ┌──▶ analyzer  ──┐              ┌──▶ executor ──┐
                     ├──▶ analyzer  ──┤              ├──▶ executor ──┤
  User ──▶ Primary ──┼──▶ planner   ──┼──▶ Primary ──┼──▶ monitor  ──┼──▶ Primary ──▶ User
                     ├──▶ validator ──┤              ├──▶ reporter ──┤
                     └──▶ optimizer ──┘              └──▶ cleaner  ──┘
```

IMPORTANT: Workflow 擴展時, 一定要避免 overload (等同於 programming 時, 把 class 寫得太大包導致耦合度過大). 將來其中一個 agent 異動可能會破壞整個 Worflow

---
我們必須要知道的是, sub-agent 負責任的對象是 `main Agent` 而非 User!! 這一點非常多人搞錯. 因此配置 agents 的時候:

- main Agent 委派任務給 agents 的時候, 視情況可能給予必要的 Prompt
- sub-agent 接收的是 main Agent Prompt (而非 User Prompt), 依照 sub Agent 自己的 System Prompt 進行任務處理, 最後再回傳給 main Agent

---
sub-agents 的優缺點

- pros: 良好的 context engineering 能夠有利於 sub-agents 的設計, 藉此能夠讓 system prompt 及 tool use 配置得更加完善, 實現高度專業化分工
- cons: sub-agents 的處理細節不容易 Debug; 此外, 強烈警告 如果在不適合的確定性任務之中盲目加入代理, 只會讓 agent performance 及系統變得更加脆弱


[One Prompt Every AGENTIC Codebase Should Have](https://www.youtube.com/watch?v=3_mwKbYvbUg)

**打算解決的問題**
往往新人進來團隊以後, 初始化專案, 可以在 local 跑起來就要花很多時間(還得安裝一堆有的沒的, 看一堆過期文檔)

**細節**
藉由一種簡單的 `just` CLI, 搭配 `Justfile`(類似 Makefile), 結合 Claude Hook(在啟動的時候, 做環境初始化, 並且記錄必要的 logs, 最後再讓 Agent 閱讀, 來協助解決問題)

- PART 1
  - 純粹傳統自動化 -- 只有 HOOK (這和 USER 直接進去專案跑腳本沒兩樣, 只是靠 AGENT 運行)
  - 僅使用 Claude Code 的 setup hook, e.g.: `claude --init` & `claude --maintenance`
- PART 2
  - 加上 AI & supervisor (HOOK + PROMPT), e.g.: `claude --init "install"` & `claude --maintenance "/maintenance"`
  - 將 Script 與 Prompt 結合, 並且將 "常見問題及解法"寫入 PROMPT
  - 賦予 AGENT 記錄及排查 logs, 並且搭配 **/prime command** 驗證是否成
  - 這階段讓初始化專案 & 維護任務變得 predictable!
- Part 3
  - 加入人機引導(HIL, Human-in-the-loop) 實現互動
  - HOOK + PROMPT + HIL, e.g.: `claude --init "/prompt true"`
  - 藉此來實踐更加客製化的部署及維運

**簡要結論**
藉由良好的搭配: `PROMPT + HOOK + JUST`, 便可以將此流程套用到所有專案, 用以打造「會自動執行的活文件(living document that executes)」


In [ ]:
// .claude/settings.json
{
  "hooks": {
    "SessionStart": [
      {
        "hooks": [
          {
            "type": "command",
            "command": "\"$CLAUDE_PROJECT_DIR\"/.claude/hooks/session_start.py",
            "timeout": 30
          }
        ]
      }
    ],
    "Setup": [
      {
        "matcher": "init",
        "hookd": [
          {
            "type": "command",
            "command": "\"$CLAUDE_PROJECT_DIR\"/.claude/hooks/setup_init.py",
            "timeout": 30
          }
        ]
      },
      {
        "matcher": "maintenance",
        "hookd": [
          {
            "type": "command",
            "command": "\"$CLAUDE_PROJECT_DIR\"/.claude/hooks/setup_maintenance.py",
            "timeout": 30
          }
        ]
      }
    ],
  },
}

✅ [I Studied Stripe's AI Agents... Vibe Coding Is Already Dead](https://www.youtube.com/watch?v=V5A1IU8VVp4)

**試圖傳授的重要觀念**
- Vibe coding 已死; 未來工程師的重心將從單純的編碼, 轉向如何引導 Agents 進行具備可追蹤性與系統化的開發
- Developers 應該要努力發展 `Meta Agent` - 用來管理(增刪改) Agents. 也就是說, 要讓 Agents 自己控管 Agents
- 舉例 Stripe 這家掌管數百億投資規模的投資機後, 如何藉由他們自幹的 *Minions Agents* 來做一套有規範的 Agents 團隊

**細節**
(沒掌握, 整部影片都在導讀 Stripe 釋出的說明文章)

**最終目標**
建立能自我管理的 Agents 生態系統

✅ [Agentic Prompt Engineering with Claude Code(For you, your team, your AGENTS)](https://www.youtube.com/watch?v=luqKnexhpFs&list=PLS_o2ayVCKvBR3jawG9JFIzJ1vXffi8fS&index=15)

**Key points**

設計良好的 Agents 有底下的階段(越下面越高級):
- Workflow Prompt - 設計 Agent 的基礎
- Control Flow Prompt - 可作條件判斷及回圈
- Delegation Prompt - 啟動 sub-Agent
- Template Meta Prompt - 讓 Agent 生成 Prompt/Agent

---
後半部沒看了, 意識到原來作者也在賣課程

✅ [Engineers… Claude Code Output Styles Are Here. Don’t Miss This TREND](https://www.youtube.com/watch?v=mJhsWrEv-Go)

**影片核心重點**
- 並不是有多重要的影片, 就一些特色而已
- Claude Code 可以配置 `Output Style` 及 `Satus Line` 用來讓 LLM 回應的格式符合使用者一開始的配置

**影片摘要**
- (2025Q3) 當時 Claude Code CLI 可藉由 `/output style` 變更讓 Claude Code 輸出的風格做變動, 當時可選擇:
  - html / yaml / table / text-to-speech / ...
- (2026Q1) 目前 Claude Code CLI 則藉由 `/config > Output style` 變更選擇不同風格:
  - Default / Explanatory / Learning

**如何使用**

- 進入 Claude Code CLI, 輸入 `/config` > `Output style`
- 進入 Claude Code CLI, 輸入 `/statusline`

🌟[Agent Experts: Finally, Agents That ACTUALLY Learn](https://www.youtube.com/watch?v=zTcDwqopvKE)

WARNING: 這個章節屬於比較進階, 然而有些觀念非常棒, 依然值得學習

**AI Agent核心問題**
- 不管 Prompt 寫得再好, 現在的 Agent 由於會 "FORGET", 代表他們並非真正的懂得 "LEARN"
- 通用智能體 v.s. 專業智能體 的核心區別:
  - generic agent - Executes and FORGETS
  - agent experts - Executes and **LEARNS**

**細節觀念**
- Agents 必須要能夠 **在適當的時機打破常規**, 並且把這些資訊, 迭代更新, 才撐得上是真正的 AI Agent
  - Memory files / Skills / Sub-Agents / Meta-Agents 都應該要能夠自動迭代更新 (而非人工介入)
- Agents 自我學習的基本工作流程: Act + Learn + Reuse -- **AT RUNTIME**
   - 真正的專家懂得自我學習/強化/適應, 懂得不斷更新 `Mental Models`

> "Self-Improving Template Meta Prompt"
>   Meta Prompt: A prompt that builds other prompts. 
>   Template: Focused on solving a specific, recurring problem.
>   Self-Improving: Automatically updates itself, related prompts, or isolated files with new information that improves your agent's next execution.

**Use cases(未完整)**

```
// prompt(commands) writing prompts
/meta_prompt create a new version of .claude/commands/question.md called question-w-mermaid-diagrams.md where we add diagrams to the answer to the question

// agent build agent
@agent-meta-agent create a planner agent that directly reads and executes the .claude/commands/plan.md prompt. Simple and concise. Pass the incoming user prompt through to  the plan using the SlashCommand tool.

// skill create skill
use the meta-skill: create 'start-orchestrator' skill that kicks off our frontend and backend of apps/orchestrator_3_stream/application in background mode by default (adjustable). it will know how to kick off the backend with specific params, make sure it reads the scripts/start_fe.sh and scripts/start_be.sh so its aware of flags (session + cwd) after the apps are running, open the ui in chrome
```

每個 Codebase 都應該要有自己的 `META Agents`

**最終目標**
- 讓 AI Agent 自我迭代更新 -- 似乎與自適應有點關聯


[RAW Agentic Coding: ZERO to Agent SKILL](https://www.youtube.com/watch?v=X2ciJedw2vU)
